In [1]:
import pandas as pd
import numpy as np
import json

import os
from dotenv import load_dotenv


load_dotenv()

dataset_path = os.getenv("DATASET_PATH")

In [2]:
practice = pd.read_csv(os.path.join(dataset_path, "practice_puzzles.csv"))
puzzles = pd.read_csv(os.path.join(dataset_path, "puzzles.csv"))
sample = pd.read_csv(os.path.join(dataset_path, "sample_submission.csv"))

In [3]:
OPS = {"+","-","*","/"}

In [4]:
def santa_calculator(
        starting_values,
        targets,
        value_limit,
        program,
        n_regs=48,
        max_instr=4096
):
    """
    Run a program on the Elf Calculator and check that it follows every rule.

    The program is executed step by step. Execution stops at the first illegal
    instruction. After the last instruction, all targets must have appeared at
    least once in some register.

    :param starting_values: Starting positive integers, loaded into registers
        0, 1, 2, ... in their listed order. All other registers start undefined.
    :param targets: Positive integers that must each appear at least once in
        some register (a value already present at the start counts as covered).
    :param value_limit: Largest value allowed in any register, including
        intermediate results.
    :param program: List of instructions, each shaped like ``[op, i, j, d]``
        where ``op`` is one of "+", "-", "*", "/", ``i`` and ``j`` are the
        source registers, and ``d`` is the destination register.
    :param n_regs: Number of available registers (indices 0 to n_regs - 1).
    :param max_instr: Maximum number of instructions allowed in a program.
    :return: A tuple ``(valid, n_instructions, message)``.
        ``valid`` is True only if every instruction is legal and every target
        was covered. ``n_instructions`` is the program length (the puzzle's
        score) when valid, or the step number where the error occurred when
        invalid. ``message`` is "ok" or a description of the problem.
    """

    if len(program) > max_instr:
        return False, len(program), f"Too long: {len(program)}> {max_instr}"

    regs = [None] * n_regs
    for k,v in enumerate(starting_values):
        regs[k] = v

    remaining = set(targets) - set(starting_values)

    for step, ins in enumerate(program, start=1):
        if not (isinstance(ins, list) and len(ins)==4):
            return False, step, f"step {step}: Wrong Instruction Format"
        op, i, j, d = ins

        if op not in OPS:
            return False, step, f"step {step}: Unknown Instruction"

        if not all (isinstance(x, int) and 0<= x < n_regs for x in (i, j, d)):
            return False, step, f"step {step}: Indeks Register Out of Range"

        a, b = regs[i], regs[j]
        if a is None or b is None:
            return False, step, f"step {step}: Register is Not Defined"

        if op == "+":
            r = a+b
        elif op == "-":
            if a <= b:
                return False, step, f"step {step}: {a}-{b} is not positive"
            r = a-b
        elif op == "*":
            r = a*b
        else:
            if a%b != 0:
                return False, step, f"step {step}: {a} is not a multiple of {b}"
            r = a//b

        if r > value_limit:
            return False, step, f"step {step}: {r} is greater than {value_limit}"

        regs[d] = r
        remaining.discard(r)

    if remaining:
        return False, len(program), f"{len(remaining)} target is not recovered"
    return True, len(program), "ok"

In [5]:
def score_submission(puzzles_df, sub_df):

    progs = dict(zip(sub_df["id"], sub_df["program"]))
    total, bad = 0, []

    for row in puzzles_df.itertuples():
        if row.id not in progs:
            bad.append((row.id, "None in Submission"))
            continue
        program = json.loads(progs[row.id])
        ok, n, msg = santa_calculator(row.starting_values, row.targets,
                                    row.value_limit, program, row.registers)
        if ok:
            total += n
        else:
            bad.append((row.id, msg))

    print(f"Skor total: {total}")
    print(f"Puzzle not valid: {len(bad)}")
    for b in bad[:5]:
        print("  ", b)
    return total, bad

In [6]:
ok = santa_calculator([3], [1, 2, 5, 8], 9,
                    [["/",0,0,1], ["-",0,1,2], ["+",0,2,3], ["+",0,3,4]])
print(ok)   # (True, 4, 'ok')


print(santa_calculator([3, 5], [1], 100, [["-",0,1,2]]))
# (False, 1, 'step 1: 3 - 5 tidak positif')

(True, 4, 'ok')
(False, 1, 'step 1: 3-5 is not positive')


In [7]:
for df in (practice, puzzles):
    if isinstance(df["starting_values"].iloc[0], str):
        df["starting_values"] = df["starting_values"].apply(json.loads)
        df["targets"] = df["targets"].apply(json.loads)

print(type(puzzles["starting_values"].iloc[0]))

<class 'list'>


In [8]:
total, bad = score_submission(puzzles, sample)

Skor total: 1490379
Puzzle not valid: 0
